# 02 — Baseline Classifiers with LOSO Cross-Validation

Evaluates Logistic Regression, Random Forest, Gradient Boosting, and SVM under Leave-One-Storm-Out (LOSO) CV, exposing the severe generalization gap on rare port disruption events.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.models.baselines import BASELINE_REGISTRY
from src.evaluation.loso_cv import run_loso_cv
from src.data.preprocessing import FEATURE_COLS, GROUP_COL, LABEL_COL

# Assumes df was prepared in notebook 01; load it here:
df = pd.read_parquet("../data/processed/portex_labeled.parquet")
# or: df = pd.read_csv("../data/processed/portex_labeled.csv")


In [ ]:
# ── Run all baselines under LOSO ────────────────────────
all_results = {}
for name, factory in BASELINE_REGISTRY.items():
    print(f"\n▶ {name}")
    model = factory()
    result = run_loso_cv(model, df, feature_cols=FEATURE_COLS,
                         label_col=LABEL_COL, group_col=GROUP_COL)
    all_results[name] = result


In [ ]:
# ── Summary table ───────────────────────────────────────
rows = []
for name, res in all_results.items():
    rows.append({"Model": name,
                 "Mean AUC": f"{res['mean_auc']:.3f}",
                 "Std AUC":  f"{res['std_auc']:.3f}",
                 "Mean F1":  f"{res['mean_f1']:.3f}"})
pd.DataFrame(rows)


In [ ]:
# ── Train vs Test AUC (overfitting evidence) ─────────────
from src.utils.visualization import plot_loso_auc_comparison
fig, ax = plt.subplots(figsize=(8, 5))
plot_loso_auc_comparison(all_results, ax=ax)
plt.savefig("../results/figures/02_baseline_auc.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Per-storm AUC distribution ──────────────────────────
fig, axes = plt.subplots(1, len(all_results), figsize=(14, 4), sharey=True)
for ax, (name, res) in zip(axes, all_results.items()):
    aucs = res["fold_results"]["auc"].dropna()
    ax.hist(aucs, bins=15, color="#607D8B", edgecolor="white", alpha=0.85)
    ax.axvline(aucs.mean(), color="crimson", linestyle="--", label=f"mean={aucs.mean():.3f}")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("AUC"); ax.legend(fontsize=8)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Per-Storm AUC Distribution (LOSO)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("../results/figures/02_per_storm_auc.png", dpi=150); plt.show()
